In [1]:
# Simple Gen AI app using LangChain and OpenAI

import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
# Langsmith tracking
os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGCHAIN_API_KEY')
os.environ['LANGCHAIN_TRACING_V2'] = "true"
os.environ['LANGCHAIN_PROJECT'] = os.getenv('LANGCHAIN_PROJECT')


```markdown
1. Ortam Değişkenlerini Yükleme
.env dosyasındaki API anahtarlarını ve ayarları yüklüyor.
OpenAI ve LangChain servislerine erişim için gerekli anahtarlar ortam değişkenlerine atanıyor.
```

In [2]:
# Data Ingestion from the website
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://docs.smith.langchain.com/administration/tutorials/manage_spend")
docs = loader.load()
docs

USER_AGENT environment variable not set, consider setting it to identify your requests.


[Document(metadata={'source': 'https://docs.smith.langchain.com/administration/tutorials/manage_spend', 'title': 'Optimize tracing spend on LangSmith | 🦜️🛠️ LangSmith', 'description': 'Before diving into this content, it might be helpful to read the following:', 'language': 'en'}, page_content='\n\n\n\n\nOptimize tracing spend on LangSmith | 🦜️🛠️ LangSmith\n\n\n\n\n\n\n\n\nSkip to main contentWe are growing and hiring for multiple roles for LangChain, LangGraph and LangSmith.  Join our team!API ReferenceRESTPythonJS/TSSearchRegionUSEUGo to AppGet StartedObservabilityEvaluationPrompt EngineeringDeployment (LangGraph Platform)AdministrationTutorialsOptimize tracing spend on LangSmithHow-to GuidesSetupConceptual GuideSelf-hostingPricingReferenceCloud architecture and scalabilityAuthz and AuthnAuthentication methodsdata_formatsEvaluationDataset transformationsRegions FAQsdk_referenceChangelogCloud architecture and scalabilityAuthz and AuthnAuthentication methodsdata_formatsEvaluationDatase

```markdown
2. Web Sitesinden Veri Çekme
Belirtilen web sayfasındaki içeriği çekiyor ve docs değişkenine kaydediyor.

```

In [3]:
# Load Data -> Split Data -> Vector Embeddings -> Create Vector Store
from langchain.text_splitter import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
texts = text_splitter.split_documents(docs)
texts

[Document(metadata={'source': 'https://docs.smith.langchain.com/administration/tutorials/manage_spend', 'title': 'Optimize tracing spend on LangSmith | 🦜️🛠️ LangSmith', 'description': 'Before diving into this content, it might be helpful to read the following:', 'language': 'en'}, page_content='Optimize tracing spend on LangSmith | 🦜️🛠️ LangSmith\n\n\n\n\n\n\n\n\nSkip to main contentWe are growing and hiring for multiple roles for LangChain, LangGraph and LangSmith.  Join our team!API ReferenceRESTPythonJS/TSSearchRegionUSEUGo to AppGet StartedObservabilityEvaluationPrompt EngineeringDeployment (LangGraph Platform)AdministrationTutorialsOptimize tracing spend on LangSmithHow-to GuidesSetupConceptual GuideSelf-hostingPricingReferenceCloud architecture and scalabilityAuthz and AuthnAuthentication methodsdata_formatsEvaluationDataset transformationsRegions FAQsdk_referenceChangelogCloud architecture and scalabilityAuthz and AuthnAuthentication methodsdata_formatsEvaluationDataset transfor

```markdown
3. Metni Parçalara Bölme
Web’den alınan metni, 1000 karakterlik parçalara (chunk) bölüyor.
Parçalar arasında 200 karakterlik örtüşme (overlap) var.
Amaç: Uzun metinleri daha yönetilebilir parçalara ayırmak.
```

In [4]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings()
from langchain.vectorstores import FAISS
vector_store_db = FAISS.from_documents(texts, embeddings)
vector_store_db


```markdown
4. Vektör Veritabanı Oluşturma
Her metin parçası için OpenAI’ın embedding (vektör) temsili oluşturuluyor.
FAISS ile bu vektörler bir veritabanında saklanıyor.
Amaç: Benzerlik araması yapabilmek.
```

In [5]:
query = "LangSmith has two usage limits: total traces and extended"
result = vector_store_db.similarity_search(query, k=3)
result[0].page_content

'Optimization 2: limit usage\u200b\nIn the previous section, we managed data retention settings to optimize existing spend. In this section, we will\nuse usage limits to prevent future overspend.\nLangSmith has two usage limits: total traces and extended retention traces. These correspond to the two metrics we\'ve\nbeen tracking on our usage graph. We can use these in tandem to have granular control over spend.\nTo set limits, we navigate back to Settings -> Usage and Billing -> Usage configuration. There is a table at the\nbottom of the page that lets you set usage limits per workspace. For each workspace, the two limits appear, along\nwith a cost estimate:\n\nLets start by setting limits on our production usage, since that is where the majority of spend comes from.\nSetting a good total traces limit\u200b\nPicking the right "total traces" limit depends on the expected load of traces that you will send to LangSmith. You should\nclearly think about your assumptions before setting a lim

```markdown
5. Sorgu ile Benzer Metinleri Bulma
Kullanıcıdan gelen sorguya en çok benzeyen 3 metin parçası bulunuyor.

```

In [6]:
# Retrieval chain , document chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o")

prompt = ChatPromptTemplate.from_template(
    """ Answer the following question based on the context provided:
<context> {context} </context>
"""
)

document_chain = create_stuff_documents_chain(
    llm=llm,    
    prompt=prompt,
    document_variable_name="context"
)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template=' Answer the following question based on the context provided:\n<context> {context} </context>\n'), additional_kwargs={})])
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x0000026F7EF51AF0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000026F7EFA7680>, root_client=<openai.OpenAI object at 0x0000026F7EF523F0>, root_async_client=<openai.AsyncOpenAI object at 0x0000026F7EF51EB0>, model_name='gpt-4o', model_kwargs={}, openai_api_key=SecretStr('**********'))
| StrOutputParser(), kwargs={}, config={'run_name

```markdown
6. LLM için Prompt ve Zincir Oluşturma
GPT-4o modelini kullanacak şekilde bir LLM (dil modeli) nesnesi oluşturuluyor.
Modelin cevap vereceği prompt (talimat) hazırlanıyor.
Zincir (chain), modelin seçilen metin parçalarını kullanarak cevap üretmesini sağlıyor.
```

In [ ]:
from langchain_core.documents import Document
document_chain.invoke({
    "input": query,
    "context": [Document(page_content=doc.page_content) for doc in result]    
})
    

'What are the two usage limits available in LangSmith for managing spend?\n\n1. **Total Traces**: This limit tracks all traces sent to LangSmith.\n2. **Extended Retention Traces**: This limit tracks traces that have the Extended 400 Day Data Retention.'

```markdown
7. Zinciri Çalıştırma
Sorgu ve ilgili metin parçaları modele veriliyor.
Model, bu bilgilerle cevap üretiyor.
```

In [11]:
# Input --> Retrieval --> vector store --> LLM
retriever = vector_store_db.as_retriever(search_kwargs={"k": 3})
from langchain.chains import create_retrieval_chain

retrieval_chain = create_retrieval_chain(
   retriever,
   document_chain
)
retrieval_chain


RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000026F5AA7CE90>, search_kwargs={'k': 3}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template=' Answer the following question based on the context provided:\n<context> {context} </context>\n'), additional_kwargs={})])
            | ChatOp

```markdown
8. Retrieval Chain (Geri Getirme Zinciri) Kurma
Vektör veritabanı bir “retriever” (getirici) olarak ayarlanıyor.
retrieval_chain: Sorgu → Benzer metinleri bul → LLM ile cevapla şeklinde otomatik bir zincir kuruyor.
```

In [13]:
# Get response from the llm
response = retrieval_chain.invoke({ 
    "input": query
})
print(response['answer'])

Based on the context provided, here is the process to set usage limits and manage costs effectively for LangSmith:

1. **Navigate to Usage Configuration:**
   - Go to `Settings -> Usage and Billing -> Usage configuration`.
   - At the bottom, you'll find a table to set usage limits per workspace.

2. **Understand Your Current Usage:**
   - Start with understanding current usage via the `Usage Graph`.
   - This graph will show how much of each usage-based metric has been consumed.
   - Use the `Invoices` tab to see a draft of the current month's invoice for running spend.

3. **Set Usage Limits:**
   - Focus on production usage first, as it's where the majority of spend occurs.
   - Determine the right "total traces" limit based on expected load and assumptions.
   - Use the table to set both "total traces" and "extended retention traces" limits.

4. **Optimize Based on Spend:**
   - Analyze the data from the graphs and invoices to fine-tune your limits.
   - Focus on environments respo

```markdown
9. Sonuçları Alma ve Yazdırma
Zincir çalıştırılıyor, modelin cevabı ve kullandığı metin parçaları ekrana yazdırılıyor.

```

In [14]:
print(response['context'])

[Document(id='94b10f6a-5a25-4f6e-bbb3-91872520d87e', metadata={'source': 'https://docs.smith.langchain.com/administration/tutorials/manage_spend', 'title': 'Optimize tracing spend on LangSmith | 🦜️🛠️ LangSmith', 'description': 'Before diving into this content, it might be helpful to read the following:', 'language': 'en'}, page_content='Optimization 2: limit usage\u200b\nIn the previous section, we managed data retention settings to optimize existing spend. In this section, we will\nuse usage limits to prevent future overspend.\nLangSmith has two usage limits: total traces and extended retention traces. These correspond to the two metrics we\'ve\nbeen tracking on our usage graph. We can use these in tandem to have granular control over spend.\nTo set limits, we navigate back to Settings -> Usage and Billing -> Usage configuration. There is a table at the\nbottom of the page that lets you set usage limits per workspace. For each workspace, the two limits appear, along\nwith a cost estim

```markdown
Özet:
Kodunuz, bir web sayfasından veri çekiyor, metni parçalara ayırıyor, bu parçaları vektör veritabanına kaydediyor. Kullanıcıdan gelen soruya en uygun metinleri bulup, bu metinlerle GPT-4o modeline cevap ürettiriyor. Sonuç olarak hem cevabı hem de kullanılan kaynak metinleri gösteriyor.
```

```markdown
Neden vektör veritabanı oluşturuyoruz?
    Amaç: Uzun bir metin veya çok sayıda doküman olduğunda, kullanıcının sorduğu soruya en alakalı (ilgili) bölümleri bulmak istiyoruz.
    Nasıl: Metinleri küçük parçalara ayırıp, her parçayı sayılarla (vektörlerle) temsil ediyoruz. Bu vektörler, parçaların anlamını matematiksel olarak gösteriyor.
    Vektör veritabanı: Bu vektörleri hızlıca arayabilmek için FAISS gibi bir veritabanında saklıyoruz.
Neden benzerlik bulmaya çalışıyoruz?
    Amaç: Kullanıcı bir soru sorduğunda, tüm metni modele vermek hem pahalı hem de verimsiz olur.
    Çözüm: Soruya en çok benzeyen (en alakalı) birkaç metin parçasını bulup, sadece bunları modele veriyoruz.
    Fayda: Model, gereksiz bilgiyle uğraşmaz, daha hızlı ve doğru cevap verir.
Bu akışta bunlara gerek var mıydı?
    Eğer: Elinizde çok kısa bir metin veya tek bir doküman varsa, vektör veritabanı ve benzerlik aramasına gerek olmayabilir.
    Amaç: Gerçek dünyada genellikle çok fazla veri olur. Kullanıcı ne sorarsa sorsun, en alakalı bilgiyi bulmak için bu yöntem çok faydalı ve verimlidir.
    Sonuç: Büyük veya çoklu dokümanlarda, bu adımlar gereklidir ve modern yapay zeka uygulamalarında standarttır.
Özet:
Vektör veritabanı ve benzerlik araması, doğru ve hızlı cevaplar için gereklidir; özellikle çok veriyle çalışırken. Küçük veriyle çalışıyorsanız, bu adımlar atlanabilir.


```